# Bimanual Planning Example Notebook

This notebook demonstrates the complete workflow for planning constrained bimanual motions using the minimal coordinates strategy.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Fix relative paths so contents of the src directory can be imported.
import sys
sys.path.append("..")
sys.path.append("../../constrained-bimanual-planning-example")

In [3]:
import numpy as np
import os
import time
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import networkx as nx
import torch

In [4]:
from pydrake.all import (
    StartMeshcat,
    CollisionCheckerParams,
    RobotDiagramBuilder,
    MeshcatVisualizerParams,
    Role,
    MeshcatVisualizer,
    Parser,
    LoadModelDirectives,
    ProcessModelDirectives,
    SceneGraphCollisionChecker,
    AutoDiffXd,
    RigidTransform_,
    IrisNp2Options,
    IrisZoOptions,
    SnoptSolver,
    IpoptSolver,
    IrisParameterizationFunction,
    MathematicalProgram,
    HPolyhedron,
    IrisNp2,
    IrisZo,
    Hyperellipsoid,
    RandomGenerator,
    ComputePairwiseIntersections,
    GcsTrajectoryOptimization,
    Point,
    GraphOfConvexSetsOptions,
    FunctionHandleTrajectory,
    InitializeAutoDiff,
    ExtractGradient,
    Toppra,
    PathParameterizedTrajectory,
    PiecewisePolynomial,
    CompositeTrajectory,
    CalcGridPointsOptions,
    BsplineBasis,
    BsplineTrajectory,
    KinematicTrajectoryOptimization,
    MinimumDistanceLowerBoundConstraint,
    PyFunctionConstraint,
    SolverOptions,
    CommonSolverOption,
    Solve,
    sqrt,
    DiagramBuilder,
    TrajectorySource,
    InverseDynamicsController,
    Demultiplexer,
)

In [5]:
# We replace the analytic_ik with the NN
import src.iiwa_analytic_ik as iiwa_analytic_ik
import src.common as common
import src.rrt as rrt
import src.shortcut as shortcut
import src.utils as utils
from src.iiwa_program import Iiwa14IKProgram
from ikflow.config import DEVICE


ikflow/config.py | Using device: 'cuda:0'


The meshcat visualization can be viewed in your browser, by opening the link that appears after running the following cell.

In [6]:
# Only run this cell once.
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


# Parameters

- `directives_file` is a scene description, giving the location of all the models in the world.
- `grasp_distance` is the distance between the end-effectors of the two robot arms.
- `GC2`, `GC4`, and `GC6` are the "global configuration parameters", describing which branch of the IK function to use.
- `q_tilde_bottom`, `q_tilde_middle`, and `q_tilde_top` are three key configurations that we plan between, represented in the parameterized coordinates.
- `seeds` is a list of configurations to use as seed points for the region generation. They were chosen by hand, using a virtual teleoperation notebook, similar to [this one](https://deepnote.com/workspace/Manipulation-ac8201a1-470a-4c77-afd0-2cc45bc229ff/project/0762b167-402a-4362-9702-7d559f0e73bb/notebook/iris_builder-3c25c10bc29d4c9493e48eaced475d03). One can also generate regions to cover a graph in configuration space (shown later in the code for RRT) or use the [clique covers approach](https://ieeexplore.ieee.org/abstract/document/10610005/).

In [7]:
directives_file = os.path.join(common.RepoDir(), "models/old_shelves.dmd.yaml")
grasp_distance = 0.6
q_tilde_bottom = np.array([-0.6430910102907225, 1.9156121024586796, -1.7968254667817805, 1.2945447141185198, -0.023834531305537934, -0.876966810663043, -1.7041643160834519, .63039, -.03541, 1.15614,  .02789, -.13999, -.11230,  .27472,  .23412])
q_tilde_middle = np.array([-0.5997312520566763, 1.489780849654964, -1.4739679827359913, 1.2905366081785483, -0.04421061906813227, -0.8793712572715165, -1.1603461715511334, 1.45])
q_tilde_top = np.array([-0.1994994216078726, 0.9140739951190965, -2.236618320862171, 0.5238879195899456, 0.7998441913611017, -1.3575398006936048, -1.0153092816310436, -.05539, -.04350, 1.07179,  .04151, -.09723,  .05341,  .05362, -.02652])

In [8]:
seeds = [
    q_tilde_bottom,
    np.array([-0.7341522021700233, 1.9192492722970935, -1.849050540687353, 1.4690188979347225, -0.022913995470214974, -0.7839567180379224, -1.735834076048031, 1.45]),
    np.array([-0.816394667473979, 1.9228828117510568, -1.9042766014076622, 1.6254903325102958, -0.020884458583263387, -0.6994788210824544, -1.773950224396859, 1.45]),
    np.array([-0.9076736984240236, 1.7999568628541147, -1.8278258357789336, 1.8976493299850326, -0.032028511314404574, -0.5492230012012871, -1.624933169711267, 1.45]),
    np.array([-0.90780384835653, 1.5443282072400564, -1.480882097408486, 1.9741581801564516, -0.07059018895327443, -0.5065618808846135, -1.1610690777465094, 1.45]),
    np.array([-0.877792385473089, 1.283945692440691, -1.1673903163525974, 1.7986279782674526, -0.08798686286997325, -0.605914842625335, -0.7496023024205761, 1.45]),
    np.array([-0.7363360141869535, 1.0835790623705088, -1.102219288049605, 1.3727471630916555, -0.07210415656873102, -0.8362237374759414, -0.6008766712030682, 1.45]),
    np.array([-0.7093225760311644, 0.8650840325295542, -1.4794100092984808, 1.2099934253928784, 0.44173726212402287, -0.9673197772349095, -0.9450827150678346, 2.0]),
    np.array([-0.5237049267440886, 0.7086764066165658, -1.9872212610757156, 1.045742737284787, 0.8594286107005795, -1.171705603794283, -1.1435157398017397, 2.41]),
    np.array([-0.37540312953312194, 0.7958305227244739, -2.112215906760149, 0.8433434932970723, 0.8316630398644385, -1.2430896040746857, -1.1077155278001196, 2.41]),
    q_tilde_top,
    np.array([-0.7686406052800139, 1.504938625148829, -1.4584578152597332, 1.655937158932382, -0.055175677810583384, -0.6834840454669682, -1.1418310479792013, 1.45]),
    q_tilde_middle,
]

In [9]:
# full_q_top = [-0.1995,  0.9141, -2.2366,  0.5239,  0.7998, -1.3575, -1.0153,  0.2416, 0.9023,  2.2897,  0.5287, -0.8637, -1.4123, -1.3454]
# plant.SetPositions(plant_context, full_q_top)
# diagram.ForcedPublish(context)

# target, _ = q_to_ee_target(full_q_top[:7])
# T_ad = np.concatenate([target.translation(), target.rotation().ToQuaternion().wxyz()]) ## T value
# vars = np.concatenate([full_q_top[7:], T_ad])
# z = program_right.reverse_inference(vars, pad = -0.0).detach().cpu().numpy().squeeze(0)

# full_q_tilde = np.concatenate([full_q_top[:7], z])

# # print(parameterization(full_q_tilde))
# print(z)





# Set Up Environment

This is general, boilerplate code that most Drake projects include. Note the usage of [`RobotDiagramBuilder`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_robot_diagram_builder.html) to construct a [`RobotDiagram`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_robot_diagram.html) and [`CollisionChecker`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_collision_checker.html), as opposed to the less specific [`DiagramBuilder`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1systems_1_1_diagram_builder.html). We specifically use a [`SceneGraphCollisionChecker`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_scene_graph_collision_checker.html).

In [9]:
params = CollisionCheckerParams()
builder = RobotDiagramBuilder(time_step=0.0)

meshcat_visual_params = MeshcatVisualizerParams()
meshcat_visual_params.delete_on_initialization_event = False
meshcat_visual_params.role = Role.kIllustration
meshcat_visual_params.prefix = "visual"
meshcat_visual = MeshcatVisualizer.AddToBuilder(
    builder.builder(), builder.scene_graph(), meshcat, meshcat_visual_params)

meshcat_collision_params = MeshcatVisualizerParams()
meshcat_collision_params.delete_on_initialization_event = False
meshcat_collision_params.role = Role.kProximity
meshcat_collision_params.prefix = "collision"
meshcat_collision_params.visible_by_default = False
meshcat_collision = MeshcatVisualizer.AddToBuilder(
    builder.builder(), builder.scene_graph(), meshcat, meshcat_collision_params)

plant = builder.plant()
parser = Parser(plant)
package_xml_path = os.path.join(common.RepoDir(), "package.xml")
parser.package_map().AddPackageXml(package_xml_path)
directives = LoadModelDirectives(directives_file)
ProcessModelDirectives(directives, parser)

params.robot_model_instances = [
    plant.GetModelInstanceByName("iiwa_left"),
    plant.GetModelInstanceByName("iiwa_right")
]

plant.Finalize()

# We export these inputs and outputs so we can wrap the RobotDiagram in a larger
# Diagram, which will include a controller, to simulate and visualize.
builder.builder().ExportInput(plant.get_actuation_input_port(), "actuation")
builder.builder().ExportOutput(plant.get_state_output_port(), "state")

diagram = builder.Build()

params.model = diagram
params.edge_step_size = 0.01
checker = SceneGraphCollisionChecker(params)

context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(context)
diagram.ForcedPublish(context)

INFO:drake:Allocating contexts to support implicit context parallelism 8


# Build Regions

## Set Up the Parameterization

Check out `src/iiwa_analytic_ik.py` for more details on the implementation of the analytic IK function itself. The key special aspect needed for this project is making it compatible with both `float` and Drake's `AutoDiffXd` scalar type (or numpy arrays of each). When called with `AutoDiffXd`, it will automatically perform forward-mode automatic differentiation. Care must be taken to return objects of the correct template type -- see [this documentation](https://drake.mit.edu/python_bindings.html#c-function-and-method-template-instantiations-in-python) for more information on how Drake handles templating in Python.

The parameterization has two parts, the callable function itself, and the [`IrisParameterizationFunction`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1planning_1_1_iris_parameterization_function.html) object, which wraps the callable and maintains some additional necessary information (input dimension and thread safety). The parameterization itself takes as input the configuration of the controlled arm and the self motion parameter of the subordinate arm, and outputs the configuration of the controlled arm concatenated with the configuration of the follower arm.

In [12]:
program_right = Iiwa14IKProgram(diagram, model_instance=plant.GetModelInstanceByName("iiwa_right"))
program_left = Iiwa14IKProgram(diagram, model_instance=plant.GetModelInstanceByName("iiwa_left"))
program_right.create_prog()


def q_to_ee_target(q):
    """
    Given leader (left) arm configuration, compute target pose for follower (right) link_7.
    
    Input: q - 7 joint angles for left arm (leader)
    Output: RigidTransform - target pose of right link_7, relative to right base frame
    """
    global grasp_distance
    
    ad = isinstance(q[0], AutoDiffXd)
    T = AutoDiffXd if ad else float
    q_full = np.zeros(14, dtype=type(q[0]))
    q_full[:7] = q 
    T_left_link7_world = program_left.fk(q_full, matrix=True)
    R_left = T_left_link7_world[:-1, :-1]
    p_left = T_left_link7_world[:-1, -1]
    ang = (180 - 2. * 68.) * np.pi / 180. 
    c, s = np.cos(ang), np.sin(ang)
    R_adjusted = R_left @ np.array([[-1, 0, 0], [0, 1, 0], [0, 0, -1]]) @ np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])
    p_target = p_left + R_adjusted @ np.array([0, 0, -grasp_distance])
    T_target_world = np.eye(4, dtype=type(q[0]))
    T_target_world[:-1, :-1] = R_adjusted
    T_target_world[:-1, -1] = p_target
    T_right_base_world = np.eye(4)
    T_right_base_world[:-1, -1] = np.array([0, 0.765, 0])
    T_target_relative = np.linalg.inv(T_right_base_world) @ T_target_world


    # utils.DrawAxes(RigidTransform_[float](T_target_relative), meshcat, name="t_target_relative")
    # utils.DrawAxes(RigidTransform_[float](T_target_world), meshcat, name="t_target_world")
    
    return RigidTransform_[T](T_target_relative), RigidTransform_[T](T_target_world)

def parameterization(q_tilde, jacobian = False):
    '''q_tilde: 
    input: [7 joints of left (leader) + 8 latent variables for right (follower) IK]
    output: [7 left joints, 7 right joints] (14 total)
    '''

    q_full = np.zeros(14)
    
    # Left arm (leader) configuration from input
    q_left = q_tilde[:7]
    
    # Compute target for right arm (follower)

    if jacobian:
        new_q_left = np.zeros(7, dtype = AutoDiffXd)
        for i in range(len(q_left)):
            deriv = np.zeros(7)
            deriv[i] = 1
            new_q_left[i] = AutoDiffXd(q_left[i], deriv)
        q_left = new_q_left

    tf_goal, _ = q_to_ee_target(q_left)

    T_ad = np.concatenate([tf_goal.translation(), tf_goal.rotation().ToQuaternion().wxyz()]) ## T value

    if jacobian:
        T_vals = np.array([t.value() for t in T_ad])
        T_grads = np.array([t.derivatives() for t in T_ad])
    else:
        T_vals = T_ad

    latent = 10 * q_tilde[7:] ## z value
    vars = np.hstack((T_vals, latent))
    
    # Solve for right arm (follower) using NN
    q_right = program_right.ik_inference(vars = vars, add_correction=False).detach().cpu().numpy()

    if jacobian:
        vars_tensor = torch.tensor(vars, dtype=torch.float32, device=DEVICE, requires_grad=True)

        jacobian = program_right.jacobian_gen(vars_tensor)
        jacobian_np = jacobian.detach().cpu().numpy()

        dq_right_dq_tilde = np.zeros((7, 15))
        dq_right_dq_tilde[:, :7] =  jacobian_np[:, :7] @ T_grads
        dq_right_dq_tilde[:, 7:] = jacobian_np[:, 7:]

        return dq_right_dq_tilde
    else:
        q_full[:7] = q_left
        q_full[7:] = q_right
        return q_full


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
URDFParser: Link size: 11
URDFParser: Joint size: 11
URDFParser: Done loading robot file /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
URDFParser: Link size: 11
URDFParser: Joint size: 11
URDFParser: Done loading robot file /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf


In [18]:
ee_target = q_to_ee_target(q_tilde_top[:7])[0]
T = np.concatenate([ee_target.translation(), ee_target.rotation().ToQuaternion().wxyz()])
print(T)

print(ee_target)

[ 0.69953836 -0.06500019  0.7301232   0.60951906  0.60936831  0.35884945
 -0.35830691]
RigidTransform(
  R=RotationMatrix([
    [0.48568646032378815, 0.8741327532380341, 0.0007693988844775652],
    [0.0005531750152296554, 0.0005728294971036416, -0.9999996829318346],
    [-0.874132916812742, 0.4856867319403129, -0.0002053328682678353],
  ]),
  p=[0.6995383606693135, -0.06500019024089931, 0.7301231997209605],
)


In [12]:
plant.SetPositions(plant_context, parameterization(q_tilde_bottom))
diagram.ForcedPublish(context)

IndexError: too many indices for tensor of dimension 1

In [ ]:
plant.SetPositions(plant_context, parameterization(q_tilde_top))
diagram.ForcedPublish(context)

In [ ]:
# idx = 4

# plant.SetPositions(plant_context, parameterization(seeds[idx]))
# diagram.ForcedPublish(context)

In [ ]:
STOP

## Planning with Bidirectional RRT

I've included in this repository a simple Python implementation of RRT and BiRRT. They rely on two oracles: a random configuration generator, and a validity checker. (The metric is assumed to be the L2 norm.) In the parameterized space, it is easy to generate a random configuration given the domain that we used for IrisNp2 and IrisZo. And we can check validity by simply checking for reachability, subordinate arm joint limit violations, and collisions. We follow this up by running randomized shortcutting (also a simple Python implementation) to improve the path a bit.

Finally, we construct a formal Drake [`Trajectory`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1trajectories_1_1_trajectory.html) object. We construct a twice-differentiable path for each segment with zero initial and final velocity using [`PiecewisePolynomial::CubicWithContinuousSecondDerivatives`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1trajectories_1_1_piecewise_polynomial.html#aba4275b536c162df6d8e2c06b4036f3a), before concatenating them as a [`CompositeTrajectory`](https://drake.mit.edu/doxygen_cxx/classdrake_1_1trajectories_1_1_composite_trajectory.html).

In [ ]:
def RandomConfig():
    q_tilde = np.zeros(15)
    q_tilde[:7] = np.random.uniform(low=iiwa_analytic_ik.iiwa_limits_lower, 
                                    high = iiwa_analytic_ik.iiwa_limits_upper)
    q_tilde[7:] = np.random.randn(8)
    return q_tilde

def ValidityChecker(q_tilde):
    q_target, world_target = q_to_ee_target(q_tilde[:7])
    q_full = parameterization(q_tilde)
    
    angle_error, d_error = utils.CalculateError(RigidTransform_[float](program_right.fk(q_full, matrix=True)), 
                                                world_target)
    if angle_error > 0.15: return False
    if d_error > 0.1: return False

    

    if (q_full[7:] < iiwa_analytic_ik.iiwa_limits_lower).any(): return False
    if (q_full[7:] > iiwa_analytic_ik.iiwa_limits_upper).any(): return False


    if not checker.CheckConfigCollisionFree(q_full): return False

    dq_right_dq_tilde = parameterization(q_tilde, jacobian = True)
    if np.linalg.svd(dq_right_dq_tilde, compute_uv=False)[0] > 500: return False



    return True

program_right.create_prog()
ValidityChecker(q_tilde_top)

True

In [ ]:
rrt_options = rrt.RRTOptions(
    step_size = 1e-2,
    check_size = 1e-3,
    max_vertices = 1e4,
    max_iters = 1e6,
    goal_sample_frequency = 0.01,
    always_swap = False
)
rrt_planner = rrt.BiRRT(RandomConfig, ValidityChecker)

np.random.seed(0)
path = rrt_planner.plan(q_tilde_bottom, q_tilde_top, rrt_options)

Iterations:   0%|          | 0/1000000 [00:00<?, ?it/s]

Vertices:   0%|          | 0/10000 [00:00<?, ?it/s]

In [ ]:
# We also run randomized shortcutting to improve the paths at least a little bit.
np.random.seed(0)
shortcut_path = shortcut.shortcut(path.copy(), ValidityChecker, num_tries=1e2, check_size=rrt_options.check_size)

  0%|          | 0/100 [00:00<?, ?it/s]

Applied 14 shortcuts


In [ ]:
# rrt_traj_segments = [
#     PiecewisePolynomial.CubicWithContinuousSecondDerivatives(
#         np.array([float(i) - 1.0, float(i)]),
#         np.array([shortcut_path[i-1], shortcut_path[i]]).T,
#         np.zeros(15),
#         np.zeros(15)
#     )
#     for i in range(1, len(shortcut_path))
# ]

# rrt_traj = CompositeTrajectory(rrt_traj_segments)

rrt_traj = PiecewisePolynomial.FirstOrderHold(
    np.arange(len(shortcut_path)),
    np.array(shortcut_path).T
)

In [ ]:
# Visualize the trajectory by sampling and animating it
import time

# Sample the trajectory at regular time intervals
t_start = rrt_traj.start_time()
t_end = rrt_traj.end_time()
num_samples = 1000
times = np.linspace(t_start, t_end, num_samples)

# Save the sampled trajectory so other notebooks or scripts can load it later.
q_tilde_samples = []
q_full_samples = []

# Animate the trajectory
for t in tqdm(times):
    q_tilde = rrt_traj.value(t).flatten()
    q_full = parameterization(q_tilde)
    q_tilde_samples.append(q_tilde.copy())
    q_full_samples.append(q_full.copy())
    plant.SetPositions(plant_context, q_full)
    diagram.ForcedPublish(context)
    time.sleep(0.01)  # Small delay to make animation viewable
    
    

q_tilde_samples = np.asarray(q_tilde_samples)
q_full_samples = np.asarray(q_full_samples)
output_dir = os.path.abspath(os.path.join(common.RepoDir(), "../ik-net-optimization/notebooks"))
np.save(os.path.join(output_dir, "rrt_q_tilde_samples.npy"), q_tilde_samples)
np.save(os.path.join(output_dir, "rrt_q_full_samples.npy"), q_full_samples)

print("Trajectory visualization complete!")

  0%|          | 0/1000 [00:00<?, ?it/s]

Trajectory visualization complete!


In [ ]:
t_start, t_end

(0.0, 432.0)

In [ ]:
for t in tqdm(times):
    # print(t)
    q_tilde = rrt_traj.value(t).flatten()
    q_full = parameterization(q_tilde)
    print(np.linalg.svd(parameterization(q_tilde, jacobian=True),compute_uv=False)[0])
    plant.SetPositions(plant_context, q_full)
    diagram.ForcedPublish(context)
    time.sleep(0.01)

  0%|          | 0/1000 [00:00<?, ?it/s]

52.70094648835861
35.028346725988385
44.019548210697415
39.42917215691881
40.10147868143908
45.108675514595554
49.57320895214488
42.76838258743779
46.344857729861936
57.04468945264429
52.519546862661876
51.20434899237187
47.99038062722882
49.98690738580808
50.70262572034031
48.1973031700309
47.01528778486626
43.81483312182997
47.26598110560952
45.30397082978047
40.97023821987959
39.89826516230283
38.52626105605656
59.810669892569514
57.38010083287344
55.25515781038069
55.78551984673583
44.89010859457364
45.28030692402577
48.567264021242096
47.523217767719466
51.904154689124674
42.79764319091658
41.906851519005755
75.22214849325782
97.24384045070539
107.2258087037328
133.4490886626832
145.37146420049288
151.49248919302315
232.63615289258843
324.2447410113339
201.2487415861794
34.75759863397915
44.24102918251267
46.469084777879175
46.22739919608119
44.121503932987316
42.402868537999176
38.7964461752104
47.327083759934645
47.904199573877634
48.561902524443695
46.0071499204776
87.710546131

In [ ]:
a = np.array([-0.010549,  -0.11093206,  0.8244909,  0.0721257,  0.00373585, 0.10605915, -0.05645414, 0.0737686 ])
b = np.array([-0.01133524, -0.11163543,  0.82417098,  0.07154014,  0.00265857,  0.10768249, -0.05793278, 0.07573626])
np.linalg.norm(a-b)

0.003378159127942319

In [ ]:
for t in tqdm(times):
    q_tilde = rrt_traj.value(t).flatten()
    q_full = parameterization(q_tilde)
    q_full_autodiff = np.full(14, AutoDiffXd(0));
    for i in range(len(q_full)):
        deriv = np.zeros(14)
        deriv[i] = 1
        q_full_autodiff[i] = AutoDiffXd(q_full[i], derivatives=deriv)
    pose = program_right.fk(q_full_autodiff)
    pose = np.hstack([pose[0], pose[1]])
    jac = np.zeros((7, 7))
    for i in range(7):
        jac[i, :] = pose[i].derivatives()[7:]
    
    print(np.linalg.svd(jac, compute_uv=False)[5])
       




  0%|          | 0/1000 [00:00<?, ?it/s]

0.18530740483216412
0.18485325800697414
0.18411173488569213
0.18377024874275041
0.1828082519589278
0.18244091411252364
0.18150963303598244
0.18093744549978596
0.18021447323268586
0.1790363229472177
0.17832963629503787
0.17693869311865099
0.17642043685765912
0.17520629343603064
0.17449024060578586
0.17342414816228643
0.1723106177605272
0.17163460472168146
0.170265429835051
0.1696916238862842
0.1682612743867983
0.16755246835009488
0.16637104705878
0.16527868770528528
0.16441756513222247
0.16287366110024346
0.16249102097162046
0.1619559947267197
0.16173522899071222
0.16130235128663922
0.16091424883235958
0.1605730260956326
0.16002922509646705
0.15978284290381278
0.15920424301711023
0.15896530778960816
0.15844275125234336
0.15803747255408923
0.1576305048779942
0.1570366987020749
0.15674744670999105
0.15604511984548639
0.15608375143052808
0.1564548270516407
0.1563867559595514
0.15613138528462148
0.1557002189799751
0.15538517282702313
0.1545906791689484
0.15427425227252792
0.1533574971574305